In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import requests

from tqdm.notebook import tqdm

### Настройка окружения

In [2]:
load_dotenv()

api_key = os.getenv('API_KEY')
base_url = os.getenv('BASE_URL')

### Основная часть

In [10]:
df = pd.read_csv('../data/processed/epi_r.csv')
df = df.drop(columns=['rating'])

In [71]:
ingredients = df.columns.values

nutritions = ['Ingredient', 'Total lipid (fat)', 'Fatty acids, total saturated', 'Cholesterol', 'Carbohydrate, by difference', 
              'Sodium, Na', 'Fiber, total dietary', 'Protein', 'Total Sugars', 'Vitamin A, IU', 'Vitamin C, total ascorbic acid',
              'Calcium, Ca', 'Iron, Fe', 'Vitamin D (D2 + D3), International Units', 'Vitamin E (alpha-tocopherol)', 'Vitamin K (Dihydrophylloquinone)',
              'Thiamin', 'Riboflavin', 'Niacin', 'Vitamin B-6', 'Folate, food', 'Vitamin B-12', 'Pantothenic acid', 'Phosphorus, P',
              'Magnesium, Mg', 'Zinc, Zn', 'Selenium, Se', 'Copper, Cu', 'Manganese, Mn', 'Potassium, K', 'Choline, total']


daily_norms = {'Total lipid (fat)': 78.0, 'Fatty acids, total saturated': 20.0, 'Cholesterol': 300.0, 'Carbohydrate, by difference': 275.0, 
              'Sodium, Na': 2300.0, 'Fiber, total dietary': 28.0, 'Protein': 50.0, 'Total Sugars': 50.0, 'Vitamin A, IU': 900.0,
              'Vitamin C, total ascorbic acid': 90.0, 'Calcium, Ca': 1300.0, 'Iron, Fe': 18.0, 'Vitamin D (D2 + D3), International Units': 20.0,
              'Vitamin E (alpha-tocopherol)': 15.0, 'Vitamin K (Dihydrophylloquinone)': 120.0, 'Thiamin': 1.2, 'Riboflavin': 1.3, 
              'Niacin': 16, 'Vitamin B-6': 1.7, 'Folate, food': 400.0, 'Vitamin B-12': 2.4, 'Pantothenic acid': 5, 'Phosphorus, P': 1250.0,
              'Magnesium, Mg': 420.0, 'Zinc, Zn': 11.0, 'Selenium, Se': 55.0, 'Copper, Cu': 0.9, 'Manganese, Mn': 2.3, 'Potassium, K': 4700.0, 
              'Choline, total': 550}

In [58]:
def search_food(ingredient):
    url = f'{base_url}?api_key={api_key}&query={ingredient}&dataType=Foundation,SR%20Legacy&pageSize=1'
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Ошибка: {response.status_code}")
        return None
    
def extract_nutrients(food_data, target_names, ingredient):
    result = {name: 0.0 for name in target_names}
    result['Ingredient'] = ingredient

    for nutrient in food_data.get('foodNutrients', []):
        name = nutrient.get('nutrientName')
        if name in target_names:
            result[name] = nutrient.get('value', 0.0)
            
    return result

In [ ]:
res = []
for ingredient in tqdm(ingredients):
    cur_result = {name: 0.0 for name in nutritions}
    cur_result['Ingredient'] = ingredient
    response = search_food(ingredient)
    if isinstance(response, dict):
        food_data = response['foods']
        if len(food_data) > 0:
            cur_result = extract_nutrients(food_data[0], nutritions, ingredient)
    res.append(cur_result)

In [78]:
res_df = pd.DataFrame(res)
for col in res_df.columns[1:]:
    res_df[col] = np.round(res_df[col] / daily_norms[col] * 100, 1)

In [82]:
res_df.to_csv('../data/processed/nutritions.csv', index=False)